
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>


# 02 - Exploring the Lab Environment

This demonstration is meant as a review for understanding data objects registered to Unity Catalog (UC) using Databricks. UC is a unified data governance solution designed to centralize and streamline the management of data, metadata, and access control across multiple Databricks workspaces. It provides interoperability across lakehouse formats like Delta lake and Apache Iceberg in addition to providing open APIs and built-in governance for data and AI applications. 

### Learning Objectives
By the end of this lesson, you should be able to:
- Identify and display available Unity Catalog objects, including catalogs, schemas, volumes, and tables within Databricks.
- Execute SQL queries to display data directly from files in cloud storage.

**References** For more additional reading and learning, check out the [official UC GitHub repository](https://github.com/unitycatalog/unitycatalog) and [this video on UC on Databricks](https://www.databricks.com/resources/demos/videos/data-governance/unity-catalog-overview).

## REQUIRED - SELECT A COMPUTE ENVIRONMENT

<div style="
  border-left: 4px solid #f44336;
  background: #ffebee;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#c62828; margin-bottom:6px; font-size: 1.1em;">Select Serverless Compute</strong>
  <div style="color:#333;">

Before starting this notebook, select the required compute environment listed below.

- **Serverless Compute, Version 5**  
![Serverless Select](https://files.training.databricks.com/binder/prod_main/data-ingestion-with-lakeflow-connect-en_us-3.1.3/images/20261001T131157Z/Data Ingestion with LakeFlow Connect/Includes/images/common/select-serverless.png)
<br></br>
  - How to select an environment version:
[AWS](https://docs.databricks.com/aws/en/compute/serverless/dependencies#-select-an-environment-version) |
[Azure](https://learn.microsoft.com/en-us/azure/databricks/compute/serverless/dependencies#-select-an-environment-version) |
[GCP](https://docs.databricks.com/gcp/en/compute/serverless/dependencies#-select-an-environment-version)

**NOTE:**  This notebook was **developed and tested using Serverless V5**. Other compute options may work but are not guaranteed to behave the same or support all features demonstrated.
  </div>
</div>



## A. Classroom Setup

1. Run the following cell to configure your working environment for this notebook.


In [0]:
%run ./Includes/Classroom-Setup-02

2. Complete the following to explore your **data_ingestion** schema using the Catalog UI on the left.

   a. In the left navigation bar, select the catalog icon:  ![Catalog Icon](https://files.training.databricks.com/binder/prod_main/data-ingestion-with-lakeflow-connect-en_us-3.1.3/images/20261001T131157Z/Data Ingestion with LakeFlow Connect/Includes/images/catalog_icon.png)

   b. Locate the catalog that starts with **labuser123_678*** (It should be your labusername) and expand the catalog. 

   c. Expand the **data_ingestion** schema (database). This is your schema for the course.

3. We want to modify our default catalog and default schema to use **labuser123_678** and our **data_ingestion** schema to avoid writing the three-level namespace every time we query and create tables in this course.

    However, before we proceed, note that each of us has a different catalog name. Your specific catalog name has been stored dynamically in the SQL variable `my_catalog` during the classroom setup script.

    Run the code below and confirm that SQL variable matches your specific catalog name (e.g., **labuser123_678**).

In [0]:
values(my_catalog)

4. Let's modify our default catalog and schema using the `USE CATALOG` and `USE SCHEMA` statements. This eliminates the need to specify the three-level name for objects (i.e., catalog.schema.object).

    - `USE CATALOG` – Sets the current catalog.

    - `USE SCHEMA` – Sets the current schema.

    **NOTE:** Since our dynamic catalog name is stored in the SQL variable `my_catalog` as a string, we will need to use the `IDENTIFIER` clause to interpret the constant string in our variable as a catalog name. The `IDENTIFIER` clause can interpret a constant string as any of the following:
    - Relation (table or view) name
    - Function name
    - Column name
    - Field name
    - Schema name
    - Catalog name

    [IDENTIFIER clause documentation](https://docs.databricks.com/aws/en/sql/language-manual/sql-ref-names-identifier-clause?language=SQL)

    Run the following cell to set and view your default catalog and schema. Confirm that your default catalog is **labuser123_678*** and your schema is **data_ingestion** (this uses the `my_catalog` variable created in the classroom setup script).

**NOTE:** Alternatively, you can simply add your catalog name without using the `IDENTIFIER` clause.


In [0]:
-- Change the default catalog/schema
USE CATALOG IDENTIFIER(my_catalog);
USE SCHEMA data_ingestion;


-- View current catalog and schema
SELECT 
  current_catalog(), 
  current_schema()

## B. Inspecting and Referencing Unity Catalog Objects

### Catalogs, Schemas, Volumes, and Tables
In Unity Catalog, all metadata is registered in a metastore. The hierarchy of database objects in any Unity Catalog metastore is divided into three levels, represented as a three-level namespace (example, `<catalog>.<schema>.<object>`) when you reference tables, views, volumes, models, and functions.


### B1. Catalogs

Use the `SHOW SCHEMAS` statement to view available schemas in the **labuser123_678*** catalog. Run the cell and view the results. Notice that your **data_ingestion** schema is within the **labuser123_678*** catalog. Since **labuser123_678*** is now the default catalog using the above command, we do not need to explicitly mention the catalog name when looking for schemas.

In [0]:
SHOW SCHEMAS;

### B2. Schemas
Run the `DESCRIBE SCHEMA EXTENDED` statement to see information about your **data_ingestion** schema (database) that was created for you within the **labuser123_678*** catalog. In the output below, your schema name is in the row called *Namespace Name*.  

**NOTE:** Remember, we are using the `IDENTIFIER` clause to dynamically reference your specific schema name in the lab, since each user will have a different schema name. Alternatively, you can type in the schema name.

In [0]:
DESCRIBE SCHEMA EXTENDED data_ingestion;

### B3. Tables
Use the `DESCRIBE TABLE EXTENDED` statement to describe the table `mytable`.

Run the cell and view the results. Notice the following:
- In the first few cells, you can see column information.
- Starting at cell 4, you can see additional **Delta Statistics Columns**.
- Starting at cell 8, you can see additional **Detailed Table Information**.

**NOTE:** Remember, we do not need to reference the three-level namespace (`catalog.schema.table`) because we set our default catalog and schema earlier.


In [0]:
DESCRIBE TABLE EXTENDED mytable

### B4. Volumes

Volumes are Unity Catalog objects that enable governance over non-tabular datasets. Volumes represent a logical volume of storage in a cloud object storage location. Volumes provide capabilities for accessing, storing, governing, and organizing files.

While tables provide governance over tabular datasets, volumes add governance over non-tabular datasets. You can use volumes to store and access files in **_any_** format, including structured, semi-structured, and unstructured data.

Databricks recommends using volumes to govern access to all non-tabular data. Like tables, volumes can be managed or external.

#### B4.1 UI Exploration

Complete the following to explore the **dbacademy_ecommerce** catalog:

1. In the left navigation bar, select the catalog icon:  ![Catalog Icon](https://files.training.databricks.com/binder/prod_main/data-ingestion-with-lakeflow-connect-en_us-3.1.3/images/20261001T131157Z/Data Ingestion with LakeFlow Connect/Includes/images/catalog_icon.png)

2. Locate the catalog called **dbacademy_ecommerce** and expand the catalog.

3. Expand the **v01** schema. Notice that this catalog contains two volumes, **delta** and **raw**.

4. Expand the **raw** volume. Notice that the volume contains a series of folders.

5. Expand the **users-historical** folder. Notice that the folder contains a series of files.


#### B4.2 Volume Exploration with SQL

Run the `DESCRIBE VOLUME` statement to return the metadata for the **dbacademy_ecommerce.v01.raw** volume. The metadata includes the volume name, schema, catalog, type, comment, owner, and more.

Notice the following:
- Under the **storage_location** column, you can see the cloud storage location for this volume.

- Under the **volume_type** column, it indicates this is a *MANAGED* volume.


In [0]:
DESCRIBE VOLUME dbacademy_ecommerce.v01.raw;

#### B4.3 List Files in a Volume


Use the `LIST` statement to list the available files in the **raw** volume's **users-historical** directory (`/Volumes/dbacademy_ecommerce/v01/raw/users-historical`) and view the results.

Notice the following:
- Ignore any file names that begin with an underscore (_). These are temporary or intermediate files used when writing files to a location.
- Scroll down in the results and expand one of the files where the **name** column begins with **part**. Confirm that this directory contains a series of Parquet files.


**NOTE:**  When interacting with data in volumes, use the path provided by Unity Catalog, which always follows this format: */Volumes/catalog_name/schema_name/volume_name/*.

For more information on exploring directories and data files managed with Unity Catalog volumes, check out the [Explore storage and find data files](https://docs.databricks.com/en/discover/files.html) documentation.


In [0]:
LIST '/Volumes/dbacademy_ecommerce/v01/raw/users-historical'

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>